<a href="https://colab.research.google.com/github/programminghistorian/jekyll/blob/gh-pages/assets/understanding-creating-word-embeddings/understanding-creating-word-embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Preparing Your Corpus

In [2]:
%pip install gensim

Note: you may need to restart the kernel to use updated packages.


In [167]:
# A good practice in programming is to place your import statements at the top of your code, and to keep them together

import re                                   # for regular expressions
import os                                   # to look up operating system-based info
import string                               # to do fancy things with strings
import glob                                 # to locate a specific file type
from pathlib import Path                    # to access files in other directories
import gensim                               # to access Word2Vec
from gensim.models import Word2Vec          # to access Gensim's flavor of Word2Vec
import pandas as pd                         # to sort and organize data
import random

In [193]:
data = ''

with open("mansfield.txt", mode='r', encoding='utf-8-sig') as file:
        data = [p.split("\n")[0] for p in file.read().split('\n\n')]
        file.close()

print(f"{len(data)} paragraphs looaded.")

1792 paragraphs looaded.


In [194]:
def clean_text(text):

    # Cleans the given text using regular expressions to split and lower-cased versions to create
    # a list of tokens for each text.
    # The function accepts a list of texts and returns a list of of lists of tokens

    # lower case
    tokens = re.split(
        r"[\s\n.]+",
        text.replace("—", " ")
        .replace("’", "")
        .replace("‘", " ")
        .replace("“", "")
        .replace("”", ""),
    )
    tokens = [t.lower() for t in tokens]

    # remove punctuation using regular expressions
    # this line of code locates the punctuation within the given text and compiles that punctuation into a single variable
    re_punc = re.compile('[%s]' % re.escape(string.punctuation))
    # this line of code substitutes the punctuation we just compiled with nothing ''
    tokens = [re_punc.sub('', token) for token in tokens]

    # only include tokens that aren't numbers
    tokens = [token for token in tokens if token.isalnum()]
    return tokens

In [195]:
# clean text from folder of text files, stored in the data variable
data_clean = []
for x in data:
    data_clean.append(clean_text(x))

In [196]:
# Check that the length of data and the length of data_clean are the same. Both numbers printed should be the same

print(len(data))
print(len(data_clean))

1792
1792


In [197]:
# check that the first item in data and the first item in data_clean are the same.
# both print statements should print the same word, with the data cleaning function applied in the second one

print(data[0].split()[0])
print(data_clean[0][0])

MANSFIELD
mansfield


In [198]:
# check that the last item in data_clean and the last item in data are the same
# both print statements should print the same word, with the data cleaning function applied in the second one

print(data[0].split()[-1])
print(data_clean[0][-1])

PARK
park


In [199]:
tokens = re.split(r'[ \n]+', data[0])
cleaned_tokens = data_clean[0]
diff = [
    f"before: \"{token}\" - after: \"{cleaned_tokens[n]}\""
    for n, token in enumerate(tokens)
    if token != cleaned_tokens[n]
]
print("\n".join(diff))

before: "MANSFIELD" - after: "mansfield"
before: "PARK" - after: "park"


In [202]:
total_tokens = 0
modified_tokens = 0
for n, text in enumerate(data):
    tokens = [
        i
        for i in re.split(
            r"[\s\n.\d°”“]+",
            text.replace("‘", " ‘").replace("—", " ").replace(".’", "."),
        )
        if i.translate(str.maketrans("", "", string.punctuation)) != ""
    ]
    cleaned_text = data_clean[n]
    total_tokens += len(tokens)

    # Debuging code
    """
    if len(tokens) != len(cleaned_text):
        print(tokens[-1], cleaned_text[-1])
        for x, token in enumerate(tokens):
            if token != cleaned_text[x]:
                print(n, x, tokens[x-1:x+2], cleaned_text[x]) """

    modified_tokens += len([t for i, t in enumerate(tokens) if t != cleaned_text[i]])

print(f'{modified_tokens*100/total_tokens:.2f}% of the tokens were modified')

18.63% of the tokens were modified


## Model Creation

In [374]:
# train the model
model = Word2Vec(sentences=data_clean, window=4, min_count=4, epochs=7, vector_size=64, sg=0)

# save the model
model.save("word2vec.model")

In [347]:
# load the model

model = Word2Vec.load("word2vec.model")

## Analysis ##


### Exploratory Queries

In [53]:
# set the word that we are checking for
word = "milk"

# if that word is in our vocabulary
if word in model.wv.key_to_index:

    # print a statement to let us know
    print("The word %s is in your model vocabulary" % word)

# otherwise, let us know that it isn't
else:
    print("%s is not in your model vocabulary" % word)

The word milk is in your model vocabulary


In [89]:
# returns a list with the top ten words used in similar contexts to the word "milk"
model.wv.most_similar("milk", topn=10)

[('cream', 0.9253600239753723),
 ('rich', 0.8451883792877197),
 ('filtered', 0.8158601522445679),
 ('gradually', 0.8079761862754822),
 ('molasses', 0.8002183437347412),
 ('sugar', 0.7891194820404053),
 ('add', 0.7821882963180542),
 ('rice', 0.7805569171905518),
 ('meal', 0.7790215015411377),
 ('stir', 0.7760800719261169)]

In [377]:
# returns the top ten most similar words to "recipe" that are dissimilar from "milk"
model.wv.most_similar(positive = ["recipe"], negative=["milk"], topn=10)

[('pies', 0.7970458269119263),
 ('lost', 0.7760193347930908),
 ('liable', 0.7698405385017395),
 ('purpose', 0.7403340935707092),
 ('any', 0.7396025061607361),
 ('windsor', 0.7337790131568909),
 ('shape', 0.730492115020752),
 ('would', 0.7246626019477844),
 ('baked', 0.7222105264663696),
 ('goose', 0.7220696210861206)]

In [ ]:
# returns the top ten most similar words to both "recipe" and "milk"
model.wv.most_similar(positive = ["recipe", "milk"], topn=10)

[('slices', 0.8297141194343567),
 ('slice', 0.827930212020874),
 ('celery', 0.8017773032188416),
 ('bunch', 0.7854905724525452),
 ('onions', 0.7851784229278564),
 ('pieces', 0.7832365036010742),
 ('slips', 0.7689660787582397),
 ('squares', 0.7574291825294495),
 ('square', 0.7517871260643005),
 ('circular', 0.7513970136642456)]

In [92]:
# returns a cosine similarity score for the two words you provide
model.wv.similarity("milk", "cream")

0.9253601

In [380]:
# returns a prediction for the other words in a set containing the words "flour," "eggs," and "cream"
model.predict_output_word([ "flour", "eggs", "cream"])

[('beat', 0.09148684),
 ('stir', 0.027633283),
 ('gradually', 0.021762544),
 ('eggs', 0.019338474),
 ('beaten', 0.017988661),
 ('add', 0.017464004),
 ('mix', 0.016914247),
 ('yolks', 0.016368037),
 ('milk', 0.009771384),
 ('light', 0.00892416)]

In [381]:
# displays the number of words in your model's vocabulary
print(len(model.wv))

2001


## Validation ##


In [81]:
dirpath = Path(r".").glob('*.model') #current directory plus only files that end in 'model'
files = dirpath
model_list = [] # a list to hold the actual models
model_filenames = []  # the filepath for the models so we know where they came from

In [82]:
# this for loop looks for files that end with ".model" loads them, and then adds those to a list
for filename in files:
    # turn the filename into a string and save it to "file_path"
    file_path = str(filename)
    print(file_path)
    # load the model with the file_path
    model = Word2Vec.load(file_path)
    # add the model to our mode_list
    model_list.append(model)
    # add the filepath to the model_filenames list
    model_filenames.append(file_path)

word2vec.model


In [83]:
# set the word that we are checking for
word = "milk"

# if that word is in our vocabulary
if word in model.wv.key_to_index:

    # print a statement to let us know
    print("The word %s is in your model vocabulary" % word)

# otherwise, let us know that it isn't
else:
    print("%s is not in your model vocabulary" % word)

The word milk is in your model vocabulary


In [84]:
# test word pairs that we are going to use to evaluate the models
test_words = [
    ("stir", "whisk"),
    ("cream", "milk"),
    ("cake", "muffin"),
    ("jam", "jelly"),
    ("reserve", "save"),
    ("bake", "cook"),
    ("wine", "vinegar"),
    ("chop", "dice"),
    ("sugar", "molasses"),
    ("wheat", "rice"),
]

In [85]:
# these for loops will go through each list, the test word list and the models list,
# and will run all the words through each model
# then the results will be added to a dataframe

# since NumPy 19.0, sometimes working with arrays of conflicting dimensions will throw a deprecation warning
# this warning does not impact the code or the results, so we're going to filter it out
# you can also specify "dtype=object" on the resulting array
# np.warnings.filterwarnings('ignore', category=np.VisibleDeprecationWarning)

# create an empty dataframe with the column headings we need
evaluation_results = pd.DataFrame(columns=['Model', 'Test Words', 'Cosine Similarity'], dtype=object)

counter = 0
# iterate though the model_list
for i in range(len(model_list)):

    # for each model in model_list, test the tuple pairs
    for x in range(len(test_words)):

        # calculate the similarity score for each tuple
        similarity_score = model_list[i].wv.similarity(*test_words[x])

        # create a temporary dataframe with the test results
        df = [model_filenames[i], test_words[x], similarity_score]

        # add the temporary dataframe to our final dataframe
        evaluation_results.loc[counter] = df
        counter += 1


# save the evaluation_results dataframe as a .csv called "word2vec_model_evaluation.csv" in our current directory
# if you want the .csv saved somewhere specific, include the filepath in the .to_csv() call
evaluation_results.to_csv('word2vec_model_evaluation.csv')

## Next Steps

Here are some resources if you would like to learn more about word vectors:

- The Women [Writers Vector Toolkit](https://wwp.northeastern.edu/lab/wwvt/index.html) is a web interface for exploring word vectors, accompanied by glossaries, sources, case studies, and sample assignments. This toolkit includes links to a [GitHub repository with RMD walkthroughs](https://github.com/NEU-DSG/wwp-public-code-share/tree/main/WordVectors) with code for training word2vec models in R, as well as [download and resources on preparing text corpora](https://wwp.northeastern.edu/lab/wwvt/resources/downloads/index.html).

- The [Women Writers Project Resources](https://wwp.northeastern.edu/outreach/resources/index.html) page has guides on: searching your corpus, corpus analysis and preparation, model validation and assessment, and other materials for working with word vectors.

- [Link to other PH tutorial in draft]



_This walkthrough was written on November 16, 2022 using Python 3.8.3 and Gensim 4.2.0_